In [16]:
import numpy as np
import pandas as pd
import glob
import os

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from datasets import Dataset

In [34]:
# read all text files into dataframe
folder_path = '../data/raw/agora/fulltext/*.txt'
all_files = glob.glob(folder_path)

data = []

for file in all_files:
    with open(file, "r") as text:
        text_content = text.read()

    source_file = os.path.basename(file)
    data.append((source_file, text_content))

df = pd.DataFrame(data, columns=['source_file', 'raw_text'])

df.head()

,source_file,raw_text
0,1053.txt,"Introduction\nIn recent months, generative AI ..."
1,1721.txt,Section 30. The Managed Care Reform and Pa...
2,6728.txt,AN ACT\n\nProviding for disclosures and safegu...
3,2566.txt,(a) In General.--In addition to amounts otherw...
4,2572.txt,119th CONGRESS\n 1st Session\n ...


In [35]:
# Format data, define target text, set K

corpus = df.head()["raw_text"].to_list()

query = df["raw_text"][5]
K=2

In [31]:
# lightweight huggingface embeddings
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Helper function to generate mean-pooled embeddings
def get_embeddings(texts):
    encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
    
    # Perform mean pooling to get a single vector per text
    token_embeddings = model_output.last_hidden_state
    attention_mask = encoded_input['attention_mask']
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [32]:
# Generate embeddings & normalize for Cosine Similarity
corpus_embeddings = get_embeddings(corpus)
query_embedding = get_embeddings([query])

corpus_embeddings = F.normalize(corpus_embeddings, p=2, dim=1)
query_embedding = F.normalize(query_embedding, p=2, dim=1)

In [33]:
# Compute PyTorch KNN (Using Cosine Similarity via Dot Product)
similarities = torch.mm(query_embedding, corpus_embeddings.transpose(0, 1)).squeeze(0)

# Extract top K largest similarity scores
topk_scores, topk_indices = torch.topk(similarities, k=K, largest=True)

# Display Results
print(f"Query: '{query}'\n")
print(f"Top {K} Nearest Neighbors:")
for score, idx in zip(topk_scores, topk_indices):
    print(f"- [Score: {score.item():.4f}] {corpus[idx.item()]}")

Query: 'THE PEOPLE OF THE STATE OF CALIFORNIA DO ENACT AS FOLLOWS:

SECTION 1. Section 22602 of the Business and Professions Code, as added by Section 1 of Chapter 677 of the Statutes of 2025, is amended to read:
22602. (a) If a reasonable person interacting with a companion chatbot would be misled to believe that the person is interacting with a human, an operator shall issue a clear and conspicuous notification indicating that the companion chatbot is artificially generated and not human.

(b) (1) An operator shall prevent a companion chatbot on its companion chatbot platform from engaging with users unless the operator maintains a protocol for preventing the production of suicidal ideation, suicide, or self-harm content to the user, including, but not limited to, by providing a notification to the user that refers the user to crisis service providers, including a suicide hotline or crisis text line, if the user expresses suicidal ideation, suicide, or self-harm.
(2) The operator sha